### You have a DATA LAKE in which the PIPELINES will INGEST DATA frequently. YOU need to PROCESS the files as soon as they ARRIVE

In [0]:
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/FileStore/rawsource/sales_data_first.csv")

df.display()

# Autoloader

In [0]:
df = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","csv")\
            .option("cloudFile.schemaEvolutionMode","addNewColumns")
            .option("cloudFiles.schemaLocation","/FileStore/rawdestination/checkpoint")\
            .load("/FileStore/rawsource")

In [0]:
df.writeStream.format("delta")\
            .option("checkpointLocation","/FileStore/rawdestination/checkpoint")\
            .trigger(processingTime = "3 seconds")\
            .option("mergeSchema",True)\
            .start("/FileStore/rawdestination/data")

In [0]:
%sql
select * from delta.`/FileStore/rawdestination/data`

**Auto Loader Final Master Table **
Use this table as your one-page revision sheet.
Component	What is it?	Why does it exist?	Stores	Created When	Used When	Question it Answers	If Missing	Real Life Analogy
Schema Location	Schema repository	Avoid schema inference every restart and support schema evolution	Column names, datatypes, schema versions, schema history	First schema inference	Stream start, schema evolution	What does my data look like?	Schema must be inferred again; schema evolution becomes difficult	Building blueprint
RocksDB	Embedded Key-Value Database	Track millions/billions of files efficiently	File path, file metadata, discovery status, processed status	During file discovery	Every trigger	Have I already processed this file?	Files may be reprocessed; scalability issues	Attendance register
Micro Batch	Processing unit	Process files in groups instead of one by one	Files belonging to a trigger	Every trigger	During execution	What files am I processing now?	No execution unit	Truck carrying packages
Offsets	Input log	Record planned work before execution	Batch input details	Before batch execution	Recovery	What should this batch process?	Spark won't know pending work	To-do list
Commits	Success log	Record completed work	Successful batch IDs	After successful execution	Recovery	What completed successfully?	Spark can't determine completed batches	Completed task list
Sources	Source metadata	Remember source configuration	Input path, source type, format	Stream initialization	Restart	Where is data coming from?	Source recovery becomes difficult	Address of warehouse
Checkpoint	Recovery package	Recover stream after failure	Offsets, commits, sources, metadata, RocksDB state	Stream start	Restart/Failure	How do I resume after failure?	Stream starts from scratch	Brain memory
________________________________________
Auto Loader Maintains 3 Types of State
State Type	Purpose	Component
Schema State	Structure of data	Schema Location
File State	File processing history	RocksDB
Execution State	Batch execution history	Offsets + Commits + Checkpoint
________________________________________
Checkpoint Internals
checkpoint/
├── offsets/
├── commits/
├── sources/
├── metadata
└── rocksdb/

Folder	Purpose	Example
offsets	Planned batches	Batch 10 should process file20,file21
commits	Successful batches	Batch 10 completed
sources	Source information	cloudFiles, ADLS path
metadata	Query details	Stream ID
rocksdb	File tracking database	file20 processed
________________________________________
File Processing Flow
Assume:
file1.json
file2.json
file3.json
arrive in ADLS.
Step	Component	Action
1	Auto Loader	Discovers files
2	RocksDB	Checks if files were seen earlier
3	RocksDB	Registers new files
4	Micro Batch	Creates Batch 0
5	Offsets	Stores Batch 0 input plan
6	Schema Location	Loads schema
7	Spark	Reads file contents
8	Delta	Writes records
9	Commits	Marks Batch 0 successful
10	RocksDB	Updates file status to processed
11	Checkpoint	Saves stream state
________________________________________
Offset vs Commit (Most Important Interview Question)
Feature	Offset	Commit
Meaning	Planned Work	Completed Work
Created	Before processing	After processing
Indicates Success	No	Yes
Represents	Input	Completion
Used For	Recovery planning	Recovery confirmation
Example	Process file1,file2	Batch completed successfully
Easy Memory
Offset = I WILL process
Commit = I HAVE processed
________________________________________
Failure Scenario
Suppose:
Before Crash
offsets

0
1
2
3
4
5
commits
commits

0
1
2
3
4
Analysis
Item	Status
Offset 5	Exists
Commit 5	Missing
Meaning:
Batch 5 started
Batch 5 did not finish
Restart Action
Re-run Batch 5
This prevents data loss.
________________________________________
RocksDB Deep Understanding
Conceptual View
Key	Value
file1.json	Processed
file2.json	Processed
file3.json	Processed
________________________________________
New File Detection
Storage:
file1
file2
file3
file4
File	RocksDB Result	Action
file1	Found	Skip
file2	Found	Skip
file3	Found	Skip
file4	Not Found	Process
________________________________________
Why RocksDB?
Without RocksDB	With RocksDB
Millions of filenames in driver memory	Stored efficiently on disk
High memory consumption	Low memory usage
Poor scalability	Handles billions of files
Risk of duplicates	Exactly-once processing
________________________________________
Schema Evolution Example
Version 1
{
"id":1,
"name":"John"
}

Column	Type
id	bigint
name	string
________________________________________
Version 2
{
"id":1,
"name":"John",
"email":"john@test.com"
}

Column	Type
id	bigint
name	string
email	string
Schema Location stores both versions.
________________________________________
Who Owns What?
Component	Managed By
Schema Location	Auto Loader
RocksDB	Auto Loader
Offsets	Structured Streaming
Commits	Structured Streaming
Sources	Structured Streaming
Checkpoint	Structured Streaming
Micro Batch	Structured Streaming
________________________________________
Ultimate Interview Summary
Component	One-Line Answer
Schema Location	Stores schema versions and schema evolution history.
RocksDB	Stores discovered and processed file metadata to avoid duplicate ingestion.
Micro Batch	Group of files processed in one trigger.
Offsets	Record what a batch is supposed to process.
Commits	Record which batches completed successfully.
Sources	Store source configuration and input metadata.
Checkpoint	Complete recovery package used to restart the stream without data loss.
Golden Memory Trick
Schema Location = What does data look like?
RocksDB = Have I seen this file before?
Offset = What should I process?
Commit = What did I successfully process?
Sources = Where is data coming from?
Checkpoint = How do I recover after failure?
Micro Batch = What am I processing right now?


### **Question 2-
### 
### You have a DATA LAKE in which the
### PIPELINES will INGEST DATA frequently
### 
### (BUT WITH NEW SCHEMA) YOU need to
### PROCESS the files as soon as they ARRIVE
### **

note -There are 3 files ..when uploading 3 files we will see this error

Auto Loader Schema Evolution - Complete Notes
1. Components Involved
Component	Purpose	Stores
Schema Location	Source schema repository	Schema versions, columns, datatypes
Cached Schema (Memory)	Schema used by running stream	Current active schema
Auto Loader	Reads files	Compares file schema with cached schema
Delta Table	Target table	Table schema
mergeSchema	Delta schema evolution	Allows new columns in target table
________________________________________
2. Initial Flow
File 1
{
"id": 1,
"name": "John"
}
File 2
{
"id": 2,
"name": "Mike"
}
Auto Loader infers:
id
name
________________________________________
Schema Stored
Schema Location (Permanent Copy)
Version 1
id
name
Cached Schema (Memory Copy)
When stream starts:
id
name
gets loaded into memory.
Schema Location
↓
Load Schema
↓
Memory Cache
________________________________________
3. New File Arrives
File 3
{
"id": 3,
"name": "David",
"email": "d@test.com"
}
File schema:
id
name
email
New column:
email
________________________________________
4. What Auto Loader Does
Auto Loader compares:
Cached Schema	File Schema
id	id
name	name
❌ email missing	✅ email exists
Schema mismatch detected.
________________________________________
5. Auto Loader Updates Schema Location
Old Schema:
Version 1
id
name
New Schema:
Version 2
id
name
email
Stored in:
schemaLocation
________________________________________
Important Point
At this time:
Schema Location
id
name
email
Memory Cache
id
name
Still old.
________________________________________
6. Why Stream Fails
Auto Loader does NOT immediately change the schema already loaded into memory.
The running query still uses:
id
name
When processing File3:
File
id
name
email
Cache
id
name
Mismatch:
email column not found
Result:
UnknownFieldException
or
Schema mismatch error
Stream stops.
________________________________________
7. Why Doesn't Auto Loader Refresh Cache Immediately?
Spark Structured Streaming works like:
Start Stream
↓
Read Schema
↓
Create Execution Plan
↓
Run Micro Batches
The execution plan is already built.
Spark cannot safely modify the schema in the middle of a running query.
Therefore:
Update schemaLocation
↓
Stop/Fail Query
↓
Restart Query
↓
Load Updated Schema
________________________________________
8. What Does Restart Mean?
Restart does NOT mean:
Restart Cluster
Restart means:
Stop Streaming Query
↓
Start Streaming Query Again
Example:
query.stop()
Then start again.
________________________________________
9. After Restart
Stream starts again.
Reads latest schema from:
schemaLocation
Current schema:
id
name
email
Loads into memory.
New Cache
id
name
email
Now:
File Schema	Cached Schema
id	id
name	name
email	email
Match ✅
Auto Loader succeeds.
________________________________________
10. Two Schema Checks Happen
This is a very important interview concept.
Check 1: Auto Loader Schema Validation
Auto Loader compares:
Incoming File Schema
VS
Cached Schema
Example:
id
name
email
VS
id
name

Failure occurs.
After restart:
id
name
email
VS
id
name
email
Success.
________________________________________
Check 2: Delta Table Schema Validation
Suppose target table:
orders
id
name
Incoming data:
id
name
email
Auto Loader is happy.
But Delta says:
I don't have email column.
Write fails.
________________________________________
11. Delta Schema Evolution
Enable:
.option("mergeSchema","true")
This allows Delta to evolve.
________________________________________
Before
Target table:
id
name
Incoming Data
id
name
email
________________________________________
After mergeSchema
Target table:
id
name
email
Write succeeds.
________________________________________
12. Who Handles What?
Layer	Responsibility
Auto Loader	Read new columns from source files
Schema Location	Store source schema history
Cached Schema	Active schema used by stream
Delta Table	Store target schema
mergeSchema	Add new columns to Delta table
________________________________________
13. Autoloader vs Delta Schema Evolution
Feature	Auto Loader	Delta Table
Purpose	Read new file columns	Update target table schema
Location	schemaLocation	Delta table metadata
Setting	cloudFiles.schemaEvolutionMode="addNewColumns"	mergeSchema=true
Handles	Source schema changes	Target schema changes
________________________________________
14. Complete End-to-End Flow
File1
(id,name)
↓
Infer Schema
↓
Schema Location
(id,name)
↓
Load Into Cache
(id,name)
↓
Write To Delta
Later:
File3
(id,name,email)
↓
Compare With Cache
(id,name)
↓
New Column Found
↓
Update schemaLocation
(id,name,email)
↓
Cache Still Old
(id,name)
↓
Mismatch
↓
Stream Stops
↓
Restart Query
↓
Reload Schema
(id,name,email)
↓
Auto Loader Success
↓
Check Delta Table
↓
Need mergeSchema=true
↓
Delta Adds email
↓
Write Success
Interview Answer (Best Version)
Auto Loader maintains schema in two places: a permanent copy in schemaLocation and a cached copy in the running stream's memory. When a file arrives with a new column, Auto Loader detects the schema change and updates schemaLocation. However, the running query still uses the old cached schema, causing a schema mismatch and stream failure. After the streaming query restarts, it reloads the latest schema from schemaLocation and can read the new column. A second schema validation then occurs at the Delta table. If the target table does not contain the new column, mergeSchema=true is required so Delta can evolve its schema and successfully write the data.


### **Auto Loader Schema Evolution Modes**

Auto Loader Schema Evolution Modes
1. addNewColumns (Default)
.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
Existing Schema
id
name
New File
id
name
email
What Happens?
1. Detect new column (email)
2. Update schemaLocation
3. Stream fails
4. Restart query
5. Load updated schema
6. Continue processing
Result
Schema Updated = Yes
Stream Fails = Yes
New Column Added = Yes
Easy Memory
Fail
↓
Update Schema
↓
Restart
↓
Continue
________________________________________
2. rescue
.option("cloudFiles.schemaEvolutionMode", "rescue")
Existing Schema
id
name
New File
id
name
email
What Happens?
1. Do not evolve schema
2. Do not fail stream
3. Store unknown columns in _rescued_data
Output
id name _rescued_data

1 John {"email":"john@test.com"}
Result
Schema Updated = No
Stream Fails = No
New Column Saved = Yes
Easy Memory
Don't Fail
Don't Evolve
Rescue Data
________________________________________
3. failOnNewColumns
.option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
New File
id
name
email
What Happens?
1. Detect new column
2. Fail immediately
3. Do not update schema
4. Require manual action
Result
Schema Updated = No
Stream Fails = Yes
Manual Fix Needed = Yes
Easy Memory
Fail
No Evolution
Manual Fix
________________________________________
4. none
.option("cloudFiles.schemaEvolutionMode", "none")
New File
id
name
email
What Happens?
1. Ignore email
2. Process only known columns
3. No schema update
4. No failure
Output
id
name
Result
Schema Updated = No
Stream Fails = No
New Column = Ignored
Easy Memory
Ignore
Don't Fail
Don't Evolve
________________________________________
Comparison
addNewColumns
  -> Update schema + Restart

rescue
  -> Store unknown columns in _rescued_data

failOnNewColumns
  -> Fail immediately

## none
##   -> Ignore unknown columns________________________________________
## If _rescued_data Already Exists In Source
Source File
{
"id": 1,
"name": "John",
"_rescued_data": "abc"
}
Auto Loader
.option("rescuedDataColumn", "_rescued_data")
Problem
Source column name = _rescued_data
Rescue column name = _rescued_data
Conflict
Solution
Use another rescue column name.
.option("rescuedDataColumn", "_extra_columns")
Result
Source Columns:
--------------
id
name
_rescued_data

Auto Loader Rescue Column:
--------------------------
_extra_columns

No conflict.

________________________________________
Interview Question
Can _rescued_data store multiple unknown columns?
Yes.
Input:
{
"id":1,
"name":"John",
"email":"a@test.com",
"phone":"9999999999"
}

Output:
_rescued_data

{
  "email":"a@test.com",
  "phone":"9999999999"
}
Can _rescued_data store nested JSON?
Yes.
Input:
{
"id":1,
"address":{
"city":"Hyd",
"zip":"500001"
}
}
Output:
{
"address":{
"city":"Hyd",
"zip":"500001"
}
}
Nested structure is preserved.
